# Candidate blocking

SQLite FTS5 provides country-partitioned name/address BM25, character trigrams, initials and numeric-address anchors. Vocabulary document frequencies limit broad queries. Deterministic reciprocal-rank fusion applies a configurable cap, saves channel provenance and flags overflow. Blank records produce empty candidates; all references remain in query_scope. Indexes and candidate sets have fingerprints and atomic completion manifests.

This lexical baseline is executable. Qwen dense rescue remains disabled pending measured recall and cost. Common-only names can miss retrieval: compare caps and per-country recall on early_stop before fixing a production cap. Full-scale index size and latency still need profiling.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import sqlite3
import heapq
from collections import defaultdict
_C, _ = _A["load_helpers"](_ROOT / "01_data_cleaning.ipynb")

def sqlite_readonly(path):
    con=sqlite3.connect(Path(path).resolve().as_uri()+"?mode=ro", uri=True)
    con.execute("PRAGMA cache_size=-131072")
    return con

def parquet_records(paths, batch_size=10000):
    import pyarrow.parquet as pq
    for path in paths:
        for batch in pq.ParquetFile(path).iter_batches(batch_size=batch_size):
            yield from batch.to_pylist()

def country_table(country):
    return "fts_" + _A["hashlib"].sha256(country.encode("utf-8")).hexdigest()[:20]

def build_retrieval_index(normalized_dir, audit_dir, output_dir, dataset, progress=None):
    if dataset not in ("train","test"):
        raise ValueError("Invalid dataset")
    out = Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    normalized = _A["require_manifest"](Path(normalized_dir)/"manifest.json","normalized")
    audit = _A["require_manifest"](Path(audit_dir)/"manifest.json","audit")
    if normalized["identity"]["audit"] != audit["fingerprint"] or not audit["country_partition_verified"]:
        raise ValueError("Normalization/audit mismatch or country audit missing")
    identity = {"normalized":normalized["fingerprint"],"audit":audit["fingerprint"],"dataset":dataset,
                "helpers":_A["load_helpers"](_ROOT/"02_blocking.ipynb")[1],
                "sqlite":sqlite3.sqlite_version,"version":"fts-v1"}
    fp = _A["stable_hash"](identity)
    manifest_path = out/"manifest.json"
    if manifest_path.exists():
        cached = _A["require_manifest"](manifest_path,"index")
        if cached["fingerprint"] == fp:
            return cached
    target = out/f"index-{fp[:16]}.sqlite"
    temporary = out/f"index-{_A['uuid'].uuid4().hex}.tmp.sqlite"
    con = sqlite3.connect(temporary)
    started = _A["time"].perf_counter()
    def emit(event):
        event.update(stage="index",dataset=dataset,fingerprint=fp,seconds=_A["time"].perf_counter()-started)
        _A["atomic_json"](out/"progress.json",event)
        if progress is not None:progress(event)
    try:
        con.execute("PRAGMA journal_mode=DELETE")
        con.execute("PRAGMA cache_size=-262144")
        con.execute("""CREATE TABLE records(entity_id TEXT NOT NULL,country TEXT,
            business_name TEXT,business_address TEXT,source INTEGER,lex_name TEXT,
            name_core TEXT,lex_address TEXT,grams TEXT,initials TEXT,numbers TEXT)""")
        # Reference records are stored for pair lookup but never inserted into external FTS.
        countries = set()
        for source in (1,2,3):
            batch = []
            source_count = 0
            for r in parquet_records(_C["normalized_paths"](normalized_dir,dataset,[source])):
                countries.add(r["country"])
                batch.append(tuple(r[k] for k in ("entity_id","country","business_name",
                    "business_address"))+(source,)+tuple(r[k] for k in
                    ("lex_name","name_core","lex_address","grams","initials","numbers")))
                if len(batch)>=10000:
                    con.executemany("INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?,?)",batch)
                    con.commit(); source_count+=len(batch); batch=[]
                    if source_count%100000==0:
                        emit({"state":"records","source":source,"rows":source_count})
            if batch:
                con.executemany("INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?,?)",batch)
                con.commit();source_count+=len(batch)
            emit({"state":"source_complete","source":source,"rows":source_count})
        emit({"state":"building_record_indexes"})
        con.execute("CREATE UNIQUE INDEX records_entity_id ON records(entity_id)")
        con.execute("CREATE INDEX record_country_source ON records(country,source)")
        counts = {}
        for country in sorted(countries):
            table = country_table(country)
            emit({"state":"indexing_country","country":country})
            con.execute(f"""CREATE VIRTUAL TABLE {table} USING fts5(
                entity_id UNINDEXED,name,address,grams,initials,numbers,tokenize='unicode61')""")
            con.execute(f"""INSERT INTO {table}
                SELECT entity_id,lex_name||' '||name_core,lex_address,grams,initials,numbers
                FROM records WHERE country=? AND source IN (2,3)""",(country,))
            con.execute(f"CREATE VIRTUAL TABLE {table}_vocab USING fts5vocab({table},'col')")
            counts[country]=con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
            con.commit()
            emit({"state":"country_complete","country":country,"rows":counts[country]})
        con.close(); con=None
        temporary.replace(target)
        result={"stage":"index","complete":True,"fingerprint":fp,"identity":identity,
                "database":target.name,"country_counts":counts,
                "files":[_A["file_entry"](target,out,checksum=False)]}
        _A["atomic_json"](manifest_path,result)
        emit({"state":"complete","rows":sum(counts.values()),"database_bytes":target.stat().st_size})
        return result
    finally:
        if con is not None: con.close()

def rare_terms(con, table, field, text, pool_size, limit=6, max_df_ratio=.02):
    """Cap union fan-out through observed document frequency, not string length alone."""
    ranked=[]
    max_df=max(50,int(pool_size*max_df_ratio))
    for term in sorted(set(text.split())):
        row=con.execute(f"SELECT doc FROM {table}_vocab WHERE term=? AND col=?",
                        (term,field)).fetchone()
        if row is not None and row[0]<=max_df:
            ranked.append((row[0],term))
    return [term for _,term in sorted(ranked)[:limit]]

def retrieve_candidates(con, record, pool_size, cap=40, channel_limit=80, rrf_constant=60):
    if not isinstance(cap,int) or cap<1 or channel_limit<cap:
        raise ValueError("Require positive cap and channel_limit >= cap")
    table=country_table(record["country"])
    def expression(field,terms):
        # All terms come from normalized alphanumeric fields; quote defensively.
        return field + ":(" + " OR ".join('"'+t.replace('"','""')+'"' for t in terms)+")"
    fields={"name":record["lex_name"]+" "+record["name_core"],
            "address":record["lex_address"],"grams":record["grams"],
            "initials":record["initials"],"numbers":record["numbers"]}
    terms={f:rare_terms(con,table,f,text,pool_size,limit=8 if f=="grams" else 6)
           for f,text in fields.items()}
    queries={}
    for field in ("name","address","grams","initials"):
        if terms[field]: queries[field]=expression(field,terms[field])
    if terms["numbers"] and terms["name"]:
        queries["address_numeric_anchors"]=expression("numbers",terms["numbers"])+" AND "+expression("name",terms["name"])
    fused=defaultdict(float); provenance=defaultdict(list)
    for channel,query in queries.items():
        # Ask FTS5 for ranked row IDs first; fetch record IDs only after top-k.
        # Cutoff ties use the immutable index order; retained ties use external ID.
        hits=con.execute(f"""SELECT rowid,rank FROM {table} WHERE {table} MATCH ?
            AND rank MATCH 'bm25(0,4,1,1,1,1)' ORDER BY rank LIMIT ?""",
            (query,channel_limit)).fetchall()
        rows=[(con.execute(f"SELECT entity_id FROM {table} WHERE rowid=?",(rowid,)).fetchone()[0],score)
              for rowid,score in hits]
        rows.sort(key=lambda row:(row[1],row[0]))
        for rank,(candidate_id,_) in enumerate(rows,1):
            fused[candidate_id]+=1/(rrf_constant+rank)
            provenance[candidate_id].append(channel)
    ordered=sorted(fused,key=lambda cid:(-fused[cid],cid))
    return [{"candidate_id":cid,"rank":i,"rrf":fused[cid],
             "channels":",".join(sorted(provenance[cid]))}
            for i,cid in enumerate(ordered[:cap],1)],len(ordered)>cap

def sqlite_to_parquet(con, sql, path, schema):
    import pyarrow as pa
    import pyarrow.parquet as pq
    cursor=con.execute(sql); names=[d[0] for d in cursor.description]
    with pq.ParquetWriter(path,schema,compression="zstd") as writer:
        while rows:=cursor.fetchmany(20000):
            writer.write_table(pa.Table.from_pylist([dict(zip(names,r)) for r in rows],schema=schema))

def build_candidates(index_dir, audit_dir, output_dir, cap=40, split=None,
                     reference_limit=None, channel_limit=80, progress=None):
    import pyarrow as pa
    index_dir,out=Path(index_dir),Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    index=_A["require_manifest"](index_dir/"manifest.json","index")
    audit=_A["require_manifest"](Path(audit_dir)/"manifest.json","audit")
    dataset=index["identity"]["dataset"]
    if index["identity"]["audit"]!=audit["fingerprint"]:
        raise ValueError("Index/audit mismatch")
    if split is not None and (dataset!="train" or split not in _A["SPLIT_NAMES"]):
        raise ValueError("Split applies only to training references")
    if reference_limit is not None and reference_limit<1: raise ValueError("Invalid limit")
    identity={"index":index["fingerprint"],"audit":audit["fingerprint"],"cap":cap,
              "channel_limit":channel_limit,"split":split,"reference_limit":reference_limit,
              "overflow":"deterministic RRF top-k; retain overflow flag",
              "helpers":_A["load_helpers"](_ROOT/"02_blocking.ipynb")[1]}
    fp=_A["stable_hash"](identity)
    if (out/"manifest.json").exists():
        old=_A["require_manifest"](out/"manifest.json","candidates")
        if old["fingerprint"]==fp:return old
    started=_A["time"].perf_counter()
    index_con=sqlite_readonly(index_dir/index["database"])
    audit_con=_A["connect_database"](Path(audit_dir)/audit["database"],read_only=True)
    temp=out/f"candidates-{_A['uuid'].uuid4().hex}.tmp.sqlite"
    con=sqlite3.connect(temp)
    try:
        con.execute("CREATE TABLE query_scope(reference_id TEXT PRIMARY KEY,country TEXT,overflow INTEGER)")
        con.execute("""CREATE TABLE candidates(reference_id TEXT,candidate_id TEXT,rank INTEGER,
                    rrf REAL,channels TEXT,PRIMARY KEY(reference_id,candidate_id))""")
        if dataset=="train":
            query="SELECT reference_id FROM splits"
            args=[]
            if split: query+=" WHERE split=?";args.append(split)
            query+=" ORDER BY md5(reference_id),reference_id"
        else:
            query="SELECT entity_id FROM records WHERE dataset='test' AND source=1 ORDER BY entity_id"
            args=[]
        if reference_limit is not None: query+=" LIMIT ?";args.append(reference_limit)
        cursor=audit_con.execute(query,args)
        n=pairs=overflows=0
        while rows:=cursor.fetchmany(1000):
            for (reference_id,) in rows:
                rec_cursor=index_con.execute("SELECT * FROM records WHERE entity_id=? AND source=1",(reference_id,))
                row=rec_cursor.fetchone()
                if row is None: raise ValueError("Reference absent from index")
                record=dict(zip([d[0] for d in rec_cursor.description],row))
                candidates,overflow=retrieve_candidates(index_con,record,
                    index["country_counts"][record["country"]],cap,channel_limit)
                con.execute("INSERT INTO query_scope VALUES (?,?,?)",(reference_id,record["country"],int(overflow)))
                con.executemany("INSERT INTO candidates VALUES (?,?,?,?,?)",
                    [(reference_id,c["candidate_id"],c["rank"],c["rrf"],c["channels"]) for c in candidates])
                n+=1;pairs+=len(candidates);overflows+=overflow
                if n%10==0:
                    event={"stage":"candidates","state":"running","references":n,"pairs":pairs,
                           "seconds":_A["time"].perf_counter()-started,"fingerprint":fp}
                    _A["atomic_json"](out/"progress.json",event)
                    if progress is not None:progress(event)
            con.commit()
        schemas={
          "query_scope":pa.schema([("reference_id",pa.string()),("country",pa.string()),("overflow",pa.int64())]),
          "candidates":pa.schema([("reference_id",pa.string()),("candidate_id",pa.string()),
              ("rank",pa.int64()),("rrf",pa.float64()),("channels",pa.string())])}
        files=[]
        for table,schema in schemas.items():
            p=out/f"{table}-{fp[:16]}.parquet";tmp=p.with_suffix(".tmp.parquet")
            sqlite_to_parquet(con,f"SELECT * FROM {table} ORDER BY reference_id"+
                              (",rank" if table=="candidates" else ""),tmp,schema)
            tmp.replace(p);files.append(_A["file_entry"](p,out))
        con.close();con=None
        target=out/f"candidates-{fp[:16]}.sqlite";temp.replace(target)
        files.append(_A["file_entry"](target,out,checksum=False))
        result={"stage":"candidates","complete":True,"fingerprint":fp,"identity":identity,
            "dataset":dataset,"database":target.name,"files":files,"references":n,"pairs":pairs,
            "overflow_references":overflows,"index_dir":str(index_dir.resolve()),
            "audit_dir":str(Path(audit_dir).resolve())}
        _A["atomic_json"](out/"manifest.json",result)
        return result
    finally:
        index_con.close();audit_con.close()
        if con is not None:con.close()

def candidate_recall(candidate_dir):
    """Complete truth denominator, including positives absent from retrieval."""
    candidate_dir=Path(candidate_dir)
    m=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    if m["dataset"]!="train":raise ValueError("Test truth is unavailable")
    audit_dir=Path(m["audit_dir"])
    audit=_A["require_manifest"](audit_dir/"manifest.json","audit")
    con=_A["connect_database"](audit_dir/audit["database"],read_only=True)
    try:
        for table in ("query_scope","candidates"):
            p=next(candidate_dir/f["path"] for f in m["files"] if f["path"].startswith(table) and f["path"].endswith(".parquet"))
            con.execute(f"CREATE TEMP VIEW {table} AS SELECT * FROM read_parquet({_A['sql_literal'](p)})")
        rows=con.execute("""WITH per_ref AS (
            SELECT q.reference_id,q.country,s.gold_count,
            (SELECT count(*) FROM candidates c JOIN truth t USING(reference_id,candidate_id)
             WHERE c.reference_id=q.reference_id) hits,
            (SELECT count(*) FROM candidates c WHERE c.reference_id=q.reference_id) candidates
            FROM query_scope q JOIN splits s USING(reference_id))
            SELECT country,count(*) refs,sum(gold_count) gold,sum(hits) hits,
            avg(CASE WHEN gold_count=0 THEN NULL ELSE hits*1.0/gold_count END) macro_recall,
            sum(candidates=0) empty_queries,avg(candidates) mean_candidates
            FROM per_ref GROUP BY country ORDER BY country""").fetchall()
        return {"countries":rows,"columns":["country","references","gold_links","retrieved_links",
                                           "macro_recall_non_singletons","empty_queries","mean_candidates"]}
    finally:con.close()


In [ ]:
def evaluate_candidate_caps(candidate_dir,caps=(10,20,40)):
    """Recall/cost curves and an optimistic matching ceiling, without injecting gold."""
    candidate_dir=Path(candidate_dir)
    manifest=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    if manifest["dataset"]!="train":raise ValueError("Gold recall is unavailable for test")
    caps=sorted(set(caps))
    if not caps or any(not isinstance(k,int) or k<1 or k>manifest["identity"]["cap"] for k in caps):
        raise ValueError("Caps must be positive and no larger than the persisted retrieval cap")
    audit_dir=Path(manifest["audit_dir"])
    audit=_A["require_manifest"](audit_dir/"manifest.json","audit",verify_hashes=False)
    con=_A["connect_database"](audit_dir/audit["database"],read_only=True)
    try:
        for table in ("query_scope","candidates"):
            p=next(candidate_dir/f["path"] for f in manifest["files"]
                   if f["path"].startswith(table) and f["path"].endswith(".parquet"))
            con.execute(f"CREATE TEMP VIEW {table} AS SELECT * FROM read_parquet({_A['sql_literal'](p)})")
        con.execute("CREATE TEMP TABLE caps(cap INTEGER)")
        con.executemany("INSERT INTO caps VALUES (?)",[(k,) for k in caps])
        cursor=con.execute("""WITH per_reference AS (
            SELECT q.reference_id,q.country,k.cap,s.gold_count,
                   count(c.candidate_id) candidate_count,
                   count(t.candidate_id) hits
            FROM query_scope q JOIN splits s USING(reference_id) CROSS JOIN caps k
            LEFT JOIN candidates c ON c.reference_id=q.reference_id AND c.rank<=k.cap
            LEFT JOIN truth t ON t.reference_id=c.reference_id AND t.candidate_id=c.candidate_id
            GROUP BY q.reference_id,q.country,k.cap,s.gold_count),
            expanded AS (SELECT * FROM per_reference UNION ALL
                SELECT reference_id,'ALL' country,cap,gold_count,candidate_count,hits FROM per_reference)
            SELECT country,cap,count(*) AS reference_count,sum(gold_count) gold_links,sum(hits) retrieved_links,
                sum(hits)*1.0/nullif(sum(gold_count),0) micro_recall,
                avg(CASE WHEN gold_count>0 THEN hits*1.0/gold_count END) macro_recall_non_singletons,
                avg(CASE WHEN gold_count=0 THEN 1.0 ELSE 1.25*hits/(0.25*gold_count+hits) END)
                    oracle_macro_f05,
                sum(candidate_count) candidate_pairs,avg(candidate_count) mean_candidates,
                quantile_cont(candidate_count,.95) p95_candidates,
                sum(candidate_count=0) empty_queries,
                sum(gold_count>0 AND hits=gold_count) fully_recalled_non_singletons
            FROM expanded GROUP BY country,cap ORDER BY country,cap""")
        names=[d[0] for d in cursor.description]
        return {"candidate_fingerprint":manifest["fingerprint"],
                "scope":manifest["identity"]["split"],"references":manifest["references"],
                "note":"Oracle F0.5 assumes a perfect matcher over retained candidates; it is not achieved model quality.",
                "rows":[dict(zip(names,row)) for row in cursor.fetchall()]}
    finally:con.close()



In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

RUN_BLOCKING = False
if RUN_BLOCKING:
    index = build_retrieval_index(ARTIFACT_DIR/"normalized", ARTIFACT_DIR/"audit",
                                 ARTIFACT_DIR/"index_train", "train")
    # Pilot cap is an experiment, not the locked production choice.
    candidates = build_candidates(ARTIFACT_DIR/"index_train", ARTIFACT_DIR/"audit",
        ARTIFACT_DIR/"candidates_early_stop", cap=40, split="early_stop", reference_limit=1000)
    print(candidate_recall(ARTIFACT_DIR/"candidates_early_stop"))
